In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab data access)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# 5-Class Emergency Acuity Triage: Feature Ablation on Hierarchical LightGBM Stacking (`models/train_oof_logistic_regression_stacking_new.ipynb`)

This notebook benchmarks the **Hierarchical LightGBM Stacking Pipeline** comparing **Full 38 Features** vs **Reduced 20 Features (Pure Arrival Vitals — Min/Max Removed)** across all **5 triage levels (`ESI 1..5`)** on the **5-variable Emergency Department dataset (`datasets/5v_cleandf.RData`)**:

### 🔬 Feature Roster Comparison
1. **Roster A: Previous Full 38-Feature Roster**:
   - Includes all 15 raw features: 7 arrival vitals/demographics + **8 stay min/max vitals** (`pulse_min`, `pulse_max`, `resp_min`, `resp_max`, `spo2_min`, `spo2_max`, `sbp_min`, `sbp_max`).
   - Includes 10 arrival vital flags, 4 vital ranges (`hr_range`, `rr_range`, `spo2_range`, `sbp_range`), 4 deviation indices, 3 clinical ratios (`shock_index`, `rox_index`, `bif`), and 2 instability ratios.
   - **Total**: $38$ features in $\mathbb{R}^{38}$.
2. **Roster B: Reduced 20-Feature Roster (Pure Arrival Vitals — Min/Max Removed)**:
   - **REMOVED**: `pulse_min`, `resp_min`, `spo2_min`, `sbp_min`, `pulse_max`, `resp_max`, `spo2_max`, `sbp_max` and all derived range/instability dynamics.
   - **RETAINED**: 7 raw arrival features (`age`, `cc_breathingdifficulty`, `gender`, `triage_vital_hr`, `triage_vital_sbp`, `triage_vital_rr`, `triage_vital_o2`) + 10 arrival vital flags + 3 pure arrival clinical indices (`shock_index`, `rox_index`, `bif`).
   - **Total**: $20$ features in $\mathbb{R}^{20}$.

### ⚖️ Layer-wise Resampling Strategy
- **Layer 1** (`ESI 1` vs `ESI 2..5`): Trained with **SMOTE** oversampling on the minority resuscitation class.
- **Layer 2** (`ESI 2, 3` vs `ESI 4, 5` on non-ESI 1 cohort): Trained with **1:1 Random Undersampling (RUS)**.
- **Layer 3A** (`ESI 2` vs `ESI 3` on emergent/urgent cohort): Trained with **1:1 Random Undersampling (RUS)**.
- **Layer 3B** (`ESI 4` vs `ESI 5` on lower acuity cohort): Trained with **1:1 Random Undersampling (RUS)**.

```mermaid
flowchart TD
    Data["Clean Complete Cases (5v_cleandf.RData)"] --> Split["3-Way Stratified Split: Train (98%), Val (1%), Test (1%)"]
    
    Split --> RosterA["Roster A: Full 38 Features (With Min/Max Stay Vitals & Ranges)"]
    RosterA --> L1A["Layer 1 LightGBM (SMOTE: ESI 1 vs 2..5)"]
    RosterA --> L2A["Layer 2 LightGBM (1:1 RUS: ESI 2,3 vs 4,5)"]
    RosterA --> L3AA["Layer 3A LightGBM (1:1 RUS: ESI 2 vs 3)"]
    RosterA --> L3BA["Layer 3B LightGBM (1:1 RUS: ESI 4 vs 5)"]
    L1A & L2A & L3AA & L3BA --> PipeA["Full Hierarchical Stacking + Logistic Meta-Learner (38 Features)"]
    
    Split --> RosterB["Roster B: Reduced 20 Features (Pure Arrival Vitals - No Min/Max)"]
    RosterB --> L1B["Layer 1 LightGBM (SMOTE: ESI 1 vs 2..5)"]
    RosterB --> L2B["Layer 2 LightGBM (1:1 RUS: ESI 2,3 vs 4,5)"]
    RosterB --> L3AB["Layer 3A LightGBM (1:1 RUS: ESI 2 vs 3)"]
    RosterB --> L3BB["Layer 3B LightGBM (1:1 RUS: ESI 4 vs 5)"]
    L1B & L2B & L3AB & L3BB --> PipeB["Full Hierarchical Stacking + Logistic Meta-Learner (20 Features)"]
    
    L1A & L1B --> L1Bench["Layer 1 Binary Benchmark (ESI 1)"]
    L2A & L2B --> L2Bench["Layer 2 Binary Benchmark (ESI 2,3 vs 4,5)"]
    L3AA & L3AB --> L3ABench["Layer 3A Binary Benchmark (ESI 2 vs 3)"]
    L3BA & L3BB --> L3BBench["Layer 3B Binary Benchmark (ESI 4 vs 5)"]
    PipeA & PipeB --> Benchmark["Holdout Test Benchmark: Per-Class Scoring Breakdown, Confusion Matrices & ROC Curves"]
```

### 📋 Benchmark Evaluation Objectives
1. **Standalone Sub-model Binary Evaluations**: `Layer 1` (ESI 1), `Layer 2` (ESI 2,3 vs 4,5), `Layer 3A` (ESI 2 vs 3), and `Layer 3B` (ESI 4 vs 5) at standard threshold ($p \ge 0.50$).
2. **Raw Hierarchical Probability Chain**: Multi-class argmax decisions directly from hierarchical tree probabilities.
3. **Calibrated Multi-Class Stacking Evaluation**: Overall 5-class triage acuity classification with the calibrated Multinomial Logistic Regression meta-learner using the exact `get_per_class_breakdown` scoring function.

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Raw Dataset, Filter Complete Cases & Stratified 3-Way Split (Train/Val/Test)
# ---------------------------------------------------------
suppressPackageStartupMessages({
  library(jsonlite)
  library(dplyr)
})

config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) config_path <- "config/triage_conf.json"
config <- fromJSON(config_path)

set.seed(config$training$random_state)

stratified_sample <- function(y, fraction, seed = 42) {
  set.seed(seed)
  idx_list <- split(seq_along(y), y)
  sampled <- unlist(lapply(idx_list, function(idx) {
    n_sample <- max(1, round(length(idx) * fraction))
    sample(idx, size = n_sample)
  }))
  return(sort(sampled))
}

data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) data_file <- paste0("../", data_file)

data_env <- new.env()
load(data_file, envir = data_env)

df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
raw_df   <- get(df_names[which.max(df_sizes)], envir = data_env)
target_col_name <- config$classes$target_col

initial_total_rows <- nrow(raw_df)
cat("========================================================================\n")
cat(sprintf("  INITIAL DATASET LOADED: %d Total Rows, %d Total Columns\n", initial_total_rows, ncol(raw_df)))
cat("========================================================================\n")
if (target_col_name %in% names(raw_df)) {
  cat("Initial ESI Target Distribution (including NAs):\n")
  print(table(raw_df[[target_col_name]], useNA = "ifany"))
  cat("------------------------------------------------------------------------\n")
}

gender_vec <- if ("gender" %in% names(raw_df)) ifelse(is.na(raw_df$gender), NA, ifelse(as.character(raw_df$gender) == "Male", 1, 0)) else rep(NA, nrow(raw_df))
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) raw_df$cc_breathingdifficulty else rep(NA, nrow(raw_df))

get_vec <- function(col_name) {
  if (col_name %in% names(raw_df)) {
    return(raw_df[[col_name]])
  } else {
    return(rep(NA, nrow(raw_df)))
  }
}

raw_esi <- as.character(raw_df[[target_col_name]])

df_master <- data.frame(
  age                     = raw_df$age,
  cc_breathingdifficulty  = cc_bd_vec,
  gender                  = gender_vec,
  triage_vital_hr         = get_vec("triage_vital_hr"),
  triage_vital_sbp        = get_vec("triage_vital_sbp"),
  triage_vital_rr         = get_vec("triage_vital_rr"),
  triage_vital_o2         = get_vec("triage_vital_o2"),
  pulse_min               = get_vec("pulse_min"),
  resp_min                = get_vec("resp_min"),
  spo2_min                = get_vec("spo2_min"),
  sbp_min                 = get_vec("sbp_min"),
  pulse_max               = get_vec("pulse_max"),
  resp_max                = get_vec("resp_max"),
  spo2_max                = get_vec("spo2_max"),
  sbp_max                 = get_vec("sbp_max"),
  target_col              = factor(raw_esi, levels = c("1", "2", "3", "4", "5"))
)

# Strictly drop any row containing at least 1 null/NA value across all 15 raw features or target
df_master <- na.omit(df_master)
df_master$target_num <- as.numeric(as.character(df_master$target_col))

clean_total_rows <- nrow(df_master)
dropped_rows     <- initial_total_rows - clean_total_rows

cat(sprintf("Missing Values Filter: Dropped %d rows with >= 1 NA feature (Retained %d Complete Cases, %.2f%%)\n", 
            dropped_rows, clean_total_rows, (clean_total_rows / initial_total_rows) * 100))
cat("Cleaned ESI Distribution (100% complete cases):\n")
print(table(df_master$target_col))
cat("------------------------------------------------------------------------\n")

# Stratified 3-Way Partitioning based on triage_conf.json
test_size <- config$training$test_size
val_size  <- config$training$val_size
seed_val  <- config$training$random_state

# 1. Extract Stratified Holdout Test Set (e.g., 1%)
idx_test <- stratified_sample(df_master$target_col, test_size, seed = seed_val)
test_df_clean  <- df_master[idx_test, ]
rem_df         <- df_master[-idx_test, ]

# 2. Extract Stratified Validation Set from remainder (e.g., 1% of total)
val_adj_fraction <- val_size / (1 - test_size)
idx_val <- stratified_sample(rem_df$target_col, val_adj_fraction, seed = seed_val + 1)
val_df_clean   <- rem_df[idx_val, ]
train_df_clean <- rem_df[-idx_val, ]

train_mat_export <- as.matrix(cbind(train_df_clean[, 1:15], target = train_df_clean$target_num))
val_mat_export   <- as.matrix(cbind(val_df_clean[, 1:15],   target = val_df_clean$target_num))
test_mat_export  <- as.matrix(cbind(test_df_clean[, 1:15],  target = test_df_clean$target_num))

cat(sprintf("3-Way Partition Complete:\n  Train Set      = %d rows (%.2f%%)\n  Validation Set = %d rows (%.2f%%)\n  Holdout Test   = %d rows (%.2f%%)\n", 
            nrow(train_mat_export), (nrow(train_mat_export) / clean_total_rows) * 100,
            nrow(val_mat_export),   (nrow(val_mat_export) / clean_total_rows) * 100,
            nrow(test_mat_export),  (nrow(test_mat_export) / clean_total_rows) * 100))
cat("========================================================================\n")

In [ ]:
# ---------------------------------------------------------
# Step 2: Feature Matrix Construction (Full 38 Features vs Reduced 20 Features)
# ---------------------------------------------------------
import os, json, pickle, time, warnings
import numpy as np, pandas as pd
from rpy2.robjects import r
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, recall_score,
    precision_score, f1_score, roc_auc_score, average_precision_score, confusion_matrix
)
import lightgbm as lgb
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
ROOT = '..' if os.path.basename(os.getcwd()) == 'models' else '.'

train_mat_in = np.array(r('train_mat_export'), dtype=np.float64)
val_mat_in   = np.array(r('val_mat_export'),   dtype=np.float64)
test_mat_in  = np.array(r('test_mat_export'),  dtype=np.float64)

raw_mat_tr  = train_mat_in[:, :15]
y_train     = train_mat_in[:, 15].astype(int)

raw_mat_val = val_mat_in[:, :15]
y_val       = val_mat_in[:, 15].astype(int)

raw_mat_ts  = test_mat_in[:, :15]
y_test      = test_mat_in[:, 15].astype(int)

# ---------------------------------------------------------
# 1. Roster A: Full 38-Feature Matrix (With Min/Max Stay Vitals & Ranges)
# ---------------------------------------------------------
def build_38_feature_matrix(raw_mat):
    N = len(raw_mat)
    X = np.zeros((N, 38), dtype=np.float64)
    X[:, :15] = raw_mat
    
    t_hr = raw_mat[:, 3]; t_sbp = raw_mat[:, 4]; t_rr = raw_mat[:, 5]; t_o2 = raw_mat[:, 6]
    pulse_min = raw_mat[:, 7]; resp_min = raw_mat[:, 8]; spo2_min = raw_mat[:, 9]; sbp_min = raw_mat[:, 10]
    pulse_max = raw_mat[:, 11]; resp_max = raw_mat[:, 12]; spo2_max = raw_mat[:, 13]; sbp_max = raw_mat[:, 14]
    
    hr_rng   = pulse_max - pulse_min
    rr_rng   = resp_max - resp_min
    spo2_rng = spo2_max - spo2_min
    sbp_rng  = sbp_max - sbp_min
    
    X[:, 15] = (t_o2 < 90).astype(float)
    X[:, 16] = ((t_o2 > 90) & (t_o2 < 94)).astype(float)
    X[:, 17] = (t_rr < 10).astype(float)
    X[:, 18] = (t_rr > 30).astype(float)
    X[:, 19] = (t_sbp <= 90).astype(float)
    X[:, 20] = (t_sbp > 220).astype(float)
    X[:, 21] = (t_hr < 40).astype(float)
    X[:, 22] = ((t_hr > 40) & (t_hr < 60)).astype(float)
    X[:, 23] = (t_hr > 150).astype(float)
    X[:, 24] = ((t_hr > 100) & (t_hr < 150)).astype(float)
    X[:, 25] = hr_rng; X[:, 26] = rr_rng; X[:, 27] = spo2_rng; X[:, 28] = sbp_rng
    X[:, 29] = t_hr / np.where(t_sbp == 0, 1.0, t_sbp)
    X[:, 30] = t_hr - hr_rng
    X[:, 31] = t_sbp - sbp_rng
    X[:, 32] = t_rr - rr_rng
    X[:, 33] = t_o2 - spo2_rng
    X[:, 34] = t_o2 / np.where(t_rr == 0, 1.0, t_rr)
    X[:, 35] = spo2_rng / np.where(spo2_max == 0, 1.0, spo2_max)
    X[:, 36] = hr_rng / (t_hr + 1.0)
    X[:, 37] = (t_rr / np.where(t_o2 == 0, 1.0, t_o2)) * 100.0
    return X

X_train_38_raw = build_38_feature_matrix(raw_mat_tr)
X_val_38_raw   = build_38_feature_matrix(raw_mat_val)
X_test_38_raw  = build_38_feature_matrix(raw_mat_ts)

cont_cols_38 = [0, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37]
scaler_38 = StandardScaler()
X_train_38 = X_train_38_raw.copy()
X_val_38   = X_val_38_raw.copy()
X_test_38  = X_test_38_raw.copy()
X_train_38[:, cont_cols_38] = scaler_38.fit_transform(X_train_38_raw[:, cont_cols_38])
X_val_38[:, cont_cols_38]   = scaler_38.transform(X_val_38_raw[:, cont_cols_38])
X_test_38[:, cont_cols_38]  = scaler_38.transform(X_test_38_raw[:, cont_cols_38])

# ---------------------------------------------------------
# 2. Roster B: Reduced 20-Feature Matrix (Min/Max Stay Vitals & Ranges Removed)
# ---------------------------------------------------------
def build_20_feature_matrix(raw_mat):
    N = len(raw_mat)
    X = np.zeros((N, 20), dtype=np.float64)
    # Extract ONLY arrival features (Indices 0..6: age, cc_bd, gender, triage_hr, triage_sbp, triage_rr, triage_o2)
    X[:, :7] = raw_mat[:, :7]
    
    t_hr = raw_mat[:, 3]; t_sbp = raw_mat[:, 4]; t_rr = raw_mat[:, 5]; t_o2 = raw_mat[:, 6]
    
    # 10 Binary Arrival Vital Flags (Indices 7..16)
    X[:, 7]  = (t_o2 < 90).astype(float)                          # is_dyspnea_total
    X[:, 8]  = ((t_o2 >= 90) & (t_o2 < 94)).astype(float)         # is_dyspnea_moderate
    X[:, 9]  = (t_rr < 10).astype(float)                          # is_bradypnea
    X[:, 10] = (t_rr > 30).astype(float)                          # is_tachypnea
    X[:, 11] = (t_sbp <= 90).astype(float)                        # is_hypotension
    X[:, 12] = (t_sbp > 220).astype(float)                        # is_hypertension
    X[:, 13] = (t_hr < 40).astype(float)                          # is_bradycardia_total
    X[:, 14] = ((t_hr >= 40) & (t_hr < 60)).astype(float)         # is_bradycardia_moderate
    X[:, 15] = (t_hr > 150).astype(float)                         # is_tachycardia_total
    X[:, 16] = ((t_hr > 100) & (t_hr <= 150)).astype(float)       # is_tachycardia_moderate
    
    # 3 Pure Arrival Clinical Composite Indices (Indices 17..19)
    X[:, 17] = t_hr / np.where(t_sbp == 0, 1.0, t_sbp)            # shock_index
    X[:, 18] = t_o2 / np.where(t_rr == 0, 1.0, t_rr)              # rox_index
    X[:, 19] = (t_rr / np.where(t_o2 == 0, 1.0, t_o2)) * 100.0   # bif
    return X

X_train_20_raw = build_20_feature_matrix(raw_mat_tr)
X_val_20_raw   = build_20_feature_matrix(raw_mat_val)
X_test_20_raw  = build_20_feature_matrix(raw_mat_ts)

cont_cols_20 = [0, 3, 4, 5, 6, 17, 18, 19]
scaler_20 = StandardScaler()
X_train_20 = X_train_20_raw.copy()
X_val_20   = X_val_20_raw.copy()
X_test_20  = X_test_20_raw.copy()
X_train_20[:, cont_cols_20] = scaler_20.fit_transform(X_train_20_raw[:, cont_cols_20])
X_val_20[:, cont_cols_20]   = scaler_20.transform(X_val_20_raw[:, cont_cols_20])
X_test_20[:, cont_cols_20]  = scaler_20.transform(X_test_20_raw[:, cont_cols_20])

feature_names_20 = [
    'age', 'cc_breathingdifficulty', 'gender', 'triage_vital_hr', 'triage_vital_sbp', 'triage_vital_rr', 'triage_vital_o2',
    'is_dyspnea_total', 'is_dyspnea_moderate', 'is_bradypnea', 'is_tachypnea', 'is_hypotension', 'is_hypertension',
    'is_bradycardia_total', 'is_bradycardia_moderate', 'is_tachycardia_total', 'is_tachycardia_moderate',
    'shock_index', 'rox_index', 'bif'
]

print("Feature Matrices Dimension Summary:")
print(f"  * Roster A (Full Previous 38 Features) : Train={X_train_38.shape}, Val={X_val_38.shape}, Test={X_test_38.shape}")
print(f"  * Roster B (Reduced 20 Features)       : Train={X_train_20.shape}, Val={X_val_20.shape}, Test={X_test_20.shape}")

In [ ]:
# ---------------------------------------------------------
# Step 3: Train Pipeline A — Hierarchical LightGBM Stacking on Full 38 Features
# ---------------------------------------------------------
print("=" * 80)
print("  TRAINING PIPELINE A: HIERARCHICAL STACKING ON FULL 38 FEATURES")
print("=" * 80)

def random_undersample_binary(X, y_bin, ratio=1.0, seed=42):
    np.random.seed(seed)
    pos_idx = np.where(y_bin == 1)[0]
    neg_idx = np.where(y_bin == 0)[0]
    n_pos = len(pos_idx)
    n_neg_keep = min(int(n_pos * ratio), len(neg_idx))
    kept_neg_idx = np.random.choice(neg_idx, size=n_neg_keep, replace=False)
    chosen_idx = np.sort(np.concatenate([pos_idx, kept_neg_idx]))
    return X[chosen_idx], y_bin[chosen_idx]

def numpy_smote(X, y_bin, seed=42):
    np.random.seed(seed)
    pos_mask = (y_bin == 1)
    neg_mask = (y_bin == 0)
    n_pos = np.sum(pos_mask)
    n_neg = np.sum(neg_mask)
    if n_pos == 0 or n_neg == 0 or n_pos == n_neg:
        return X, y_bin
    if n_pos < n_neg:
        min_X = X[pos_mask]; target_syn = n_neg - n_pos; min_label = 1
    else:
        min_X = X[neg_mask]; target_syn = n_pos - n_neg; min_label = 0
    n_min = len(min_X)
    syn_X = np.zeros((target_syn, X.shape[1]))
    for i in range(target_syn):
        idx1 = np.random.randint(0, n_min)
        idx2 = np.random.randint(0, n_min)
        alpha = np.random.rand()
        syn_X[i] = min_X[idx1] + alpha * (min_X[idx2] - min_X[idx1])
    return np.vstack([X, syn_X]), np.hstack([y_bin, np.full(target_syn, min_label)])

lgb_params = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'max_depth': 6,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 1,
    'verbosity': -1,
    'random_state': 42
}

t0 = time.time()

# Layer 1: ESI 1 vs (ESI 2..5) trained with SMOTE
X_sm1_38, y_sm1_38 = numpy_smote(X_train_38, (y_train == 1).astype(int))
l1_38 = lgb.LGBMClassifier(**lgb_params, n_estimators=100)
l1_38.fit(X_sm1_38, y_sm1_38, eval_set=[(X_val_38, (y_val == 1).astype(int))], callbacks=[lgb.early_stopping(10, verbose=False)])

# Layer 2: ESI 2,3 vs ESI 4,5 on non-ESI 1 with 1:1 Random Undersampling
m2_tr  = (y_train != 1); m2_val = (y_val != 1)
X_rus2_38, y_rus2_38 = random_undersample_binary(X_train_38[m2_tr], np.isin(y_train[m2_tr], [2, 3]).astype(int), ratio=1.0)
l2_38 = lgb.LGBMClassifier(**lgb_params, n_estimators=100)
l2_38.fit(X_rus2_38, y_rus2_38, eval_set=[(X_val_38[m2_val], np.isin(y_val[m2_val], [2, 3]).astype(int))], callbacks=[lgb.early_stopping(10, verbose=False)])

# Layer 3A: ESI 2 vs ESI 3 on ESI 2,3 with 1:1 Random Undersampling
m3a_tr  = np.isin(y_train, [2, 3]); m3a_val = np.isin(y_val, [2, 3])
X_rus3a_38, y_rus3a_38 = random_undersample_binary(X_train_38[m3a_tr], (y_train[m3a_tr] == 2).astype(int), ratio=1.0)
l3a_38 = lgb.LGBMClassifier(**lgb_params, n_estimators=100)
l3a_38.fit(X_rus3a_38, y_rus3a_38, eval_set=[(X_val_38[m3a_val], (y_val[m3a_val] == 2).astype(int))], callbacks=[lgb.early_stopping(10, verbose=False)])

# Layer 3B: ESI 4 vs ESI 5 on ESI 4,5 with 1:1 Random Undersampling
m3b_tr  = np.isin(y_train, [4, 5]); m3b_val = np.isin(y_val, [4, 5])
X_rus3b_38, y_rus3b_38 = random_undersample_binary(X_train_38[m3b_tr], (y_train[m3b_tr] == 4).astype(int), ratio=1.0)
l3b_38 = lgb.LGBMClassifier(**lgb_params, n_estimators=100)
l3b_38.fit(X_rus3b_38, y_rus3b_38, eval_set=[(X_val_38[m3b_val], (y_val[m3b_val] == 4).astype(int))], callbacks=[lgb.early_stopping(10, verbose=False)])

print(f"✓ Full 38-Feature Stacking sub-models trained in {time.time()-t0:.1f}s!")

def compute_hierarchical_probs(l1, l2, l3a, l3b, X_in):
    p1  = l1.predict_proba(X_in)[:, 1]
    p2  = l2.predict_proba(X_in)[:, 1]
    p3a = l3a.predict_proba(X_in)[:, 1]
    p3b = l3b.predict_proba(X_in)[:, 1]
    
    P = np.zeros((len(X_in), 5))
    P[:, 0] = p1
    P[:, 1] = (1 - p1) * p2 * p3a
    P[:, 2] = (1 - p1) * p2 * (1 - p3a)
    P[:, 3] = (1 - p1) * (1 - p2) * p3b
    P[:, 4] = (1 - p1) * (1 - p2) * (1 - p3b)
    return P

val_probs_38  = compute_hierarchical_probs(l1_38, l2_38, l3a_38, l3b_38, X_val_38)
test_probs_38 = compute_hierarchical_probs(l1_38, l2_38, l3a_38, l3b_38, X_test_38)

meta_38 = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
meta_38.fit(val_probs_38, y_val)

preds_38 = meta_38.predict(test_probs_38)
probs_38 = meta_38.predict_proba(test_probs_38)
print("✓ Pipeline A (Full 38 Features) Meta-Learner calibrated!")

In [ ]:
# ---------------------------------------------------------
# Step 4: Train Pipeline B — Hierarchical LightGBM Stacking on Reduced 20 Features (No Min/Max Vitals)
# ---------------------------------------------------------
print("=" * 80)
print("  TRAINING PIPELINE B: HIERARCHICAL STACKING ON REDUCED 20 FEATURES (NO MIN/MAX)")
print("=" * 80)

t0 = time.time()

# Layer 1: ESI 1 vs (ESI 2..5) trained with SMOTE
X_sm1_20, y_sm1_20 = numpy_smote(X_train_20, (y_train == 1).astype(int))
l1_20 = lgb.LGBMClassifier(**lgb_params, n_estimators=100)
l1_20.fit(X_sm1_20, y_sm1_20, eval_set=[(X_val_20, (y_val == 1).astype(int))], callbacks=[lgb.early_stopping(10, verbose=False)])

# Layer 2: ESI 2,3 vs ESI 4,5 on non-ESI 1 with 1:1 Random Undersampling
X_rus2_20, y_rus2_20 = random_undersample_binary(X_train_20[m2_tr], np.isin(y_train[m2_tr], [2, 3]).astype(int), ratio=1.0)
l2_20 = lgb.LGBMClassifier(**lgb_params, n_estimators=100)
l2_20.fit(X_rus2_20, y_rus2_20, eval_set=[(X_val_20[m2_val], np.isin(y_val[m2_val], [2, 3]).astype(int))], callbacks=[lgb.early_stopping(10, verbose=False)])

# Layer 3A: ESI 2 vs ESI 3 on ESI 2,3 with 1:1 Random Undersampling
X_rus3a_20, y_rus3a_20 = random_undersample_binary(X_train_20[m3a_tr], (y_train[m3a_tr] == 2).astype(int), ratio=1.0)
l3a_20 = lgb.LGBMClassifier(**lgb_params, n_estimators=100)
l3a_20.fit(X_rus3a_20, y_rus3a_20, eval_set=[(X_val_20[m3a_val], (y_val[m3a_val] == 2).astype(int))], callbacks=[lgb.early_stopping(10, verbose=False)])

# Layer 3B: ESI 4 vs ESI 5 on ESI 4,5 with 1:1 Random Undersampling
X_rus3b_20, y_rus3b_20 = random_undersample_binary(X_train_20[m3b_tr], (y_train[m3b_tr] == 4).astype(int), ratio=1.0)
l3b_20 = lgb.LGBMClassifier(**lgb_params, n_estimators=100)
l3b_20.fit(X_rus3b_20, y_rus3b_20, eval_set=[(X_val_20[m3b_val], (y_val[m3b_val] == 4).astype(int))], callbacks=[lgb.early_stopping(10, verbose=False)])

print(f"✓ Reduced 20-Feature Stacking sub-models trained in {time.time()-t0:.1f}s!")

val_probs_20  = compute_hierarchical_probs(l1_20, l2_20, l3a_20, l3b_20, X_val_20)
test_probs_20 = compute_hierarchical_probs(l1_20, l2_20, l3a_20, l3b_20, X_test_20)

meta_20 = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
meta_20.fit(val_probs_20, y_val)

preds_20 = meta_20.predict(test_probs_20)
probs_20 = meta_20.predict_proba(test_probs_20)
print("✓ Pipeline B (Reduced 20 Features) Meta-Learner calibrated!")

In [ ]:
# ---------------------------------------------------------
# Step 5: Holdout Test Set Evaluation & Scoring (from train_oof_logistic_regression_stacking.ipynb)
# ---------------------------------------------------------
def get_per_class_breakdown(y_true, y_pred, probs, pipeline_name):
    classes = [1, 2, 3, 4, 5]
    rows = []
    recalls, specs, bal_accs, aucs = [], [], [], []
    for idx, cls in enumerate(classes):
        y_bin_true = (y_true == cls).astype(int)
        y_bin_pred = (y_pred == cls).astype(int)
        tp = np.sum((y_bin_true == 1) & (y_bin_pred == 1))
        fn = np.sum((y_bin_true == 1) & (y_bin_pred == 0))
        fp = np.sum((y_bin_true == 0) & (y_bin_pred == 1))
        tn = np.sum((y_bin_true == 0) & (y_bin_pred == 0))
        rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        bal  = (rec + spec) / 2.0
        try:
            auc = roc_auc_score(y_bin_true, probs[:, idx])
        except Exception:
            auc = 0.0
        recalls.append(rec); specs.append(spec); bal_accs.append(bal); aucs.append(auc)
        rows.append({
            'Pipeline': pipeline_name,
            'Class': f'ESI_{cls}',
            'Recall': round(rec, 4),
            'Specificity': round(spec, 4),
            'Balanced_Accuracy': round(bal, 4),
            'ROC_AUC': round(auc, 4)
        })
    rows.append({
        'Pipeline': pipeline_name,
        'Class': 'Macro_Average',
        'Recall': round(np.mean(recalls), 4),
        'Specificity': round(np.mean(specs), 4),
        'Balanced_Accuracy': round(np.mean(bal_accs), 4),
        'ROC_AUC': round(np.mean(aucs), 4)
    })
    return pd.DataFrame(rows)

def evaluate_binary_submodel(y_true, y_pred, y_prob, model_name, pos_label_name="Positive"):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    bal_acc = (sens + spec) / 2.0
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    f1 = 2 * (prec * sens) / (prec + sens) if (prec + sens) > 0 else 0.0
    try: auc_val = roc_auc_score(y_true, y_prob)
    except Exception: auc_val = 0.0
    try: pr_auc = average_precision_score(y_true, y_prob)
    except Exception: pr_auc = 0.0
    
    return {
        'Model': model_name,
        f'Recall_Sensitivity_{pos_label_name}': round(sens, 4),
        'Specificity_Negative': round(spec, 4),
        'Balanced_Accuracy': round(bal_acc, 4),
        'Precision_PPV': round(prec, 4),
        'F1_Score': round(f1, 4),
        'ROC_AUC': round(auc_val, 4),
        'PR_AUC': round(pr_auc, 4),
        'TP': int(tp), 'FP': int(fp), 'TN': int(tn), 'FN': int(fn)
    }

# ---------------------------------------------------------
# Part A: STANDALONE LAYER 1 BINARY EVALUATION (ESI 1 vs Non-ESI 1, WITHOUT LogReg)
# ---------------------------------------------------------
y_test_l1 = (y_test == 1).astype(int)
prob_l1_38 = l1_38.predict_proba(X_test_38)[:, 1]
pred_l1_38 = (prob_l1_38 >= 0.50).astype(int)

prob_l1_20 = l1_20.predict_proba(X_test_20)[:, 1]
pred_l1_20 = (prob_l1_20 >= 0.50).astype(int)

l1_res = [
    evaluate_binary_submodel(y_test_l1, pred_l1_38, prob_l1_38, 'Layer 1 Full 38 Features (SMOTE, p>=0.50)', pos_label_name='ESI1'),
    evaluate_binary_submodel(y_test_l1, pred_l1_20, prob_l1_20, 'Layer 1 Reduced 20 Features (SMOTE, p>=0.50)', pos_label_name='ESI1')
]
l1_report_df = pd.DataFrame(l1_res)

print("=" * 110)
print("   STANDALONE LAYER 1 BINARY EVALUATION (ESI 1 vs Non-ESI 1, WITHOUT LOGISTIC REGRESSOR)")
print("=" * 110)
print(l1_report_df[['Model', 'Recall_Sensitivity_ESI1', 'Specificity_Negative', 'Balanced_Accuracy', 'Precision_PPV', 'F1_Score', 'ROC_AUC', 'PR_AUC']].to_string(index=False))
print("-" * 110)
print(f"Layer 1 Confusion Breakdown (Full 38)    : TP={l1_res[0]['TP']}, FP={l1_res[0]['FP']}, TN={l1_res[0]['TN']}, FN={l1_res[0]['FN']}")
print(f"Layer 1 Confusion Breakdown (Reduced 20) : TP={l1_res[1]['TP']}, FP={l1_res[1]['FP']}, TN={l1_res[1]['TN']}, FN={l1_res[1]['FN']}")
print("=" * 110 + chr(10))

# ---------------------------------------------------------
# Part B: STANDALONE LAYER 2 BINARY EVALUATION (ESI 2,3 vs ESI 4,5 on Non-ESI 1, WITHOUT LogReg)
# ---------------------------------------------------------
m2_ts = (y_test != 1)
y_test_l2 = np.isin(y_test[m2_ts], [2, 3]).astype(int)

prob_l2_38 = l2_38.predict_proba(X_test_38[m2_ts])[:, 1]
pred_l2_38 = (prob_l2_38 >= 0.50).astype(int)

prob_l2_20 = l2_20.predict_proba(X_test_20[m2_ts])[:, 1]
pred_l2_20 = (prob_l2_20 >= 0.50).astype(int)

l2_res = [
    evaluate_binary_submodel(y_test_l2, pred_l2_38, prob_l2_38, 'Layer 2 Full 38 Features (1:1 RUS, p>=0.50)', pos_label_name='ESI23'),
    evaluate_binary_submodel(y_test_l2, pred_l2_20, prob_l2_20, 'Layer 2 Reduced 20 Features (1:1 RUS, p>=0.50)', pos_label_name='ESI23')
]
l2_report_df = pd.DataFrame(l2_res)

print("=" * 110)
print("   STANDALONE LAYER 2 BINARY EVALUATION (ESI 2,3 vs ESI 4,5 on Non-ESI 1, WITHOUT LOGREG)")
print("=" * 110)
print(l2_report_df[['Model', 'Recall_Sensitivity_ESI23', 'Specificity_Negative', 'Balanced_Accuracy', 'Precision_PPV', 'F1_Score', 'ROC_AUC', 'PR_AUC']].to_string(index=False))
print("-" * 110)
print(f"Layer 2 Confusion Breakdown (Full 38)    : TP={l2_res[0]['TP']}, FP={l2_res[0]['FP']}, TN={l2_res[0]['TN']}, FN={l2_res[0]['FN']}")
print(f"Layer 2 Confusion Breakdown (Reduced 20) : TP={l2_res[1]['TP']}, FP={l2_res[1]['FP']}, TN={l2_res[1]['TN']}, FN={l2_res[1]['FN']}")
print("=" * 110 + chr(10))

# ---------------------------------------------------------
# Part C: STANDALONE LAYER 3A BINARY EVALUATION (ESI 2 vs ESI 3 on Urgent/Emergent, WITHOUT LogReg)
# ---------------------------------------------------------
m3a_ts = np.isin(y_test, [2, 3])
y_test_l3a = (y_test[m3a_ts] == 2).astype(int)

prob_l3a_38 = l3a_38.predict_proba(X_test_38[m3a_ts])[:, 1]
pred_l3a_38 = (prob_l3a_38 >= 0.50).astype(int)

prob_l3a_20 = l3a_20.predict_proba(X_test_20[m3a_ts])[:, 1]
pred_l3a_20 = (prob_l3a_20 >= 0.50).astype(int)

l3a_res = [
    evaluate_binary_submodel(y_test_l3a, pred_l3a_38, prob_l3a_38, 'Layer 3A Full 38 Features (1:1 RUS, p>=0.50)', pos_label_name='ESI2'),
    evaluate_binary_submodel(y_test_l3a, pred_l3a_20, prob_l3a_20, 'Layer 3A Reduced 20 Features (1:1 RUS, p>=0.50)', pos_label_name='ESI2')
]
l3a_report_df = pd.DataFrame(l3a_res)

print("=" * 115)
print("   STANDALONE LAYER 3A BINARY EVALUATION (ESI 2 vs ESI 3 on Urgent/Emergent, WITHOUT LOGREG)")
print("=" * 115)
print(l3a_report_df[['Model', 'Recall_Sensitivity_ESI2', 'Specificity_Negative', 'Balanced_Accuracy', 'Precision_PPV', 'F1_Score', 'ROC_AUC', 'PR_AUC']].to_string(index=False))
print("-" * 115)
print(f"Layer 3A Confusion Breakdown (Full 38)    : TP={l3a_res[0]['TP']}, FP={l3a_res[0]['FP']}, TN={l3a_res[0]['TN']}, FN={l3a_res[0]['FN']}")
print(f"Layer 3A Confusion Breakdown (Reduced 20) : TP={l3a_res[1]['TP']}, FP={l3a_res[1]['FP']}, TN={l3a_res[1]['TN']}, FN={l3a_res[1]['FN']}")
print("=" * 115 + chr(10))

# ---------------------------------------------------------
# Part D: STANDALONE LAYER 3B BINARY EVALUATION (ESI 4 vs ESI 5 on Lower Acuity, WITHOUT LogReg)
# ---------------------------------------------------------
m3b_ts = np.isin(y_test, [4, 5])
y_test_l3b = (y_test[m3b_ts] == 4).astype(int)

prob_l3b_38 = l3b_38.predict_proba(X_test_38[m3b_ts])[:, 1]
pred_l3b_38 = (prob_l3b_38 >= 0.50).astype(int)

prob_l3b_20 = l3b_20.predict_proba(X_test_20[m3b_ts])[:, 1]
pred_l3b_20 = (prob_l3b_20 >= 0.50).astype(int)

l3b_res = [
    evaluate_binary_submodel(y_test_l3b, pred_l3b_38, prob_l3b_38, 'Layer 3B Full 38 Features (1:1 RUS, p>=0.50)', pos_label_name='ESI4'),
    evaluate_binary_submodel(y_test_l3b, pred_l3b_20, prob_l3b_20, 'Layer 3B Reduced 20 Features (1:1 RUS, p>=0.50)', pos_label_name='ESI4')
]
l3b_report_df = pd.DataFrame(l3b_res)

print("=" * 115)
print("   STANDALONE LAYER 3B BINARY EVALUATION (ESI 4 vs ESI 5 on Lower Acuity, WITHOUT LOGREG)")
print("=" * 115)
print(l3b_report_df[['Model', 'Recall_Sensitivity_ESI4', 'Specificity_Negative', 'Balanced_Accuracy', 'Precision_PPV', 'F1_Score', 'ROC_AUC', 'PR_AUC']].to_string(index=False))
print("-" * 115)
print(f"Layer 3B Confusion Breakdown (Full 38)    : TP={l3b_res[0]['TP']}, FP={l3b_res[0]['FP']}, TN={l3b_res[0]['TN']}, FN={l3b_res[0]['FN']}")
print(f"Layer 3B Confusion Breakdown (Reduced 20) : TP={l3b_res[1]['TP']}, FP={l3b_res[1]['FP']}, TN={l3b_res[1]['TN']}, FN={l3b_res[1]['FN']}")
print("=" * 115 + chr(10))

# ---------------------------------------------------------
# Part E: RAW HIERARCHICAL PROBABILITY CHAIN (WITHOUT LOGISTIC REGRESSOR)
# ---------------------------------------------------------
preds_raw_chain_38 = np.argmax(test_probs_38, axis=1) + 1
preds_raw_chain_20 = np.argmax(test_probs_20, axis=1) + 1

report_raw_38 = get_per_class_breakdown(y_test, preds_raw_chain_38, test_probs_38, 'Raw_Chain_Full_38_Features (No LogReg)')
report_raw_20 = get_per_class_breakdown(y_test, preds_raw_chain_20, test_probs_20, 'Raw_Chain_Reduced_20_Features (No LogReg)')

print("=" * 95)
print("   RAW HIERARCHICAL CHAIN (WITHOUT LOGISTIC REGRESSOR) — FULL 38 FEATURES")
print("=" * 95)
print(report_raw_38.to_string(index=False))
print("=" * 95 + chr(10))

print("=" * 95)
print("   RAW HIERARCHICAL CHAIN (WITHOUT LOGISTIC REGRESSOR) — REDUCED 20 FEATURES")
print("=" * 95)
print(report_raw_20.to_string(index=False))
print("=" * 95 + chr(10))

# ---------------------------------------------------------
# Part F: FINAL CALIBRATED STACKING PIPELINE (WITH LOGISTIC REGRESSOR)
# ---------------------------------------------------------
report_38 = get_per_class_breakdown(y_test, preds_38, probs_38, 'Full_38_Features_Stacking (With LogReg)')
report_20 = get_per_class_breakdown(y_test, preds_20, probs_20, 'Reduced_20_Features_Stacking (With LogReg)')

print("=" * 95)
print("   FINAL STACKING PIPELINE (WITH LOGISTIC REGRESSOR) — FULL 38-FEATURE ROSTER")
print("=" * 95)
print(report_38.to_string(index=False))
print("=" * 95 + chr(10))

print("=" * 95)
print("   FINAL STACKING PIPELINE (WITH LOGISTIC REGRESSOR) — REDUCED 20-FEATURE ROSTER")
print("=" * 95)
print(report_20.to_string(index=False))
print("=" * 95 + chr(10))

# ---------------------------------------------------------
# Part G: CONSOLIDATED ABLATION COMPARISON TABLE
# ---------------------------------------------------------
comp_rows = []
for i in range(len(report_38)):
    cls_label = report_38.loc[i, 'Class']
    r_38, r_20 = report_38.loc[i, 'Recall'], report_20.loc[i, 'Recall']
    s_38, s_20 = report_38.loc[i, 'Specificity'], report_20.loc[i, 'Specificity']
    b_38, b_20 = report_38.loc[i, 'Balanced_Accuracy'], report_20.loc[i, 'Balanced_Accuracy']
    a_38, a_20 = report_38.loc[i, 'ROC_AUC'], report_20.loc[i, 'ROC_AUC']
    
    comp_rows.append({
        'Class': cls_label,
        'Full_38_Recall': f"{r_38*100:.2f}%",
        'Red_20_Recall': f"{r_20*100:.2f}%",
        'Delta_Recall': f"{(r_20 - r_38)*100:+.2f}%",
        'Full_38_Spec': f"{s_38*100:.2f}%",
        'Red_20_Spec': f"{s_20*100:.2f}%",
        'Full_38_BalAcc': f"{b_38*100:.2f}%",
        'Red_20_BalAcc': f"{b_20*100:.2f}%",
        'Delta_BalAcc': f"{(b_20 - b_38)*100:+.2f}%",
        'Full_38_AUC': a_38,
        'Red_20_AUC': a_20,
        'Delta_AUC': round(a_20 - a_38, 4)
    })

comp_df = pd.DataFrame(comp_rows)
print("=" * 115)
print("     CONSOLIDATED ABLATION COMPARISON: FULL 38 FEATURES vs REDUCED 20 FEATURES")
print("=" * 115)
print(comp_df.to_string(index=False))
print("=" * 115 + chr(10))

# Export reports
reports_dir = f'{ROOT}/reports'
os.makedirs(reports_dir, exist_ok=True)

l1_report_df.to_csv(os.path.join(reports_dir, 'oof_stacking_layer1_binary_report.csv'), index=False)
l2_report_df.to_csv(os.path.join(reports_dir, 'oof_stacking_layer2_binary_report.csv'), index=False)
l3a_report_df.to_csv(os.path.join(reports_dir, 'oof_stacking_layer3a_binary_report.csv'), index=False)
l3b_report_df.to_csv(os.path.join(reports_dir, 'oof_stacking_layer3b_binary_report.csv'), index=False)
report_raw_38.to_csv(os.path.join(reports_dir, 'oof_stacking_raw_chain_full_38_report.csv'), index=False)
report_raw_20.to_csv(os.path.join(reports_dir, 'oof_stacking_raw_chain_reduced_20_report.csv'), index=False)
report_38.to_csv(os.path.join(reports_dir, 'oof_stacking_full_38_report.csv'), index=False)
report_20.to_csv(os.path.join(reports_dir, 'oof_stacking_reduced_20_report.csv'), index=False)
comp_df.to_csv(os.path.join(reports_dir, 'oof_stacking_feature_ablation_comparison.csv'), index=False)
print(f"✓ All comparison, Layer 1, Layer 2, Layer 3A, and Layer 3B reports successfully exported to {reports_dir}/")

In [ ]:
# ---------------------------------------------------------
# Step 6: Side-by-Side 5x5 Confusion Matrix Comparison (Full 38 vs Reduced 20)
# ---------------------------------------------------------
plots_dir = f"{ROOT}/plots"
os.makedirs(plots_dir, exist_ok=True)
os.makedirs(os.path.join(plots_dir, 'image'), exist_ok=True)

esi_labels = [f"ESI {i}" for i in range(1, 6)]

# Compute confusion matrices
cm_38      = confusion_matrix(y_test, preds_38, labels=[1, 2, 3, 4, 5])
cm_38_norm = cm_38.astype('float') / cm_38.sum(axis=1)[:, np.newaxis]

cm_20      = confusion_matrix(y_test, preds_20, labels=[1, 2, 3, 4, 5])
cm_20_norm = cm_20.astype('float') / cm_20.sum(axis=1)[:, np.newaxis]

fig, axes = plt.subplots(1, 2, figsize=(18, 7.5))

# Left: Full 38-Feature Stacking
annot_38 = np.empty_like(cm_38, dtype=object)
for i in range(5):
    for j in range(5):
        annot_38[i, j] = f"{cm_38[i, j]:,}\n({cm_38_norm[i, j]*100:.1f}%)"

sns.heatmap(
    cm_38_norm, annot=annot_38, fmt='', cmap='Blues', cbar=True, ax=axes[0],
    vmin=0, vmax=1, xticklabels=esi_labels, yticklabels=esi_labels
)
axes[0].set_title(
    f"Pipeline A: Full 38-Feature Stacking (With Min/Max Stay Vitals)\n"
    f"Macro Balanced Acc: {report_38.loc[5, 'Balanced_Accuracy']*100:.2f}% | Macro ROC-AUC: {report_38.loc[5, 'ROC_AUC']:.4f}",
    fontsize=11.5, fontweight='bold', pad=12
)
axes[0].set_xlabel("Predicted ESI Level", fontsize=11, fontweight='bold')
axes[0].set_ylabel("True ESI Level", fontsize=11, fontweight='bold')

# Right: Reduced 20-Feature Stacking
annot_20 = np.empty_like(cm_20, dtype=object)
for i in range(5):
    for j in range(5):
        annot_20[i, j] = f"{cm_20[i, j]:,}\n({cm_20_norm[i, j]*100:.1f}%)"

sns.heatmap(
    cm_20_norm, annot=annot_20, fmt='', cmap='Greens', cbar=True, ax=axes[1],
    vmin=0, vmax=1, xticklabels=esi_labels, yticklabels=esi_labels
)
axes[1].set_title(
    f"Pipeline B: Reduced 20-Feature Stacking (Pure Arrival Vitals - No Min/Max)\n"
    f"Macro Balanced Acc: {report_20.loc[5, 'Balanced_Accuracy']*100:.2f}% | Macro ROC-AUC: {report_20.loc[5, 'ROC_AUC']:.4f}",
    fontsize=11.5, fontweight='bold', pad=12
)
axes[1].set_xlabel("Predicted ESI Level", fontsize=11, fontweight='bold')
axes[1].set_ylabel("True ESI Level", fontsize=11, fontweight='bold')

plt.tight_layout()
cm_comp_path = os.path.join(plots_dir, "oof_stacking_feature_ablation_confusion_matrix.png")
plt.savefig(cm_comp_path, dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(plots_dir, 'image', 'oof_stacking_feature_ablation_confusion_matrix.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Side-by-side Confusion Matrix comparison saved to: {cm_comp_path}")

In [ ]:
# ---------------------------------------------------------
# Step 7: Side-by-Side Multiclass ROC-AUC Curves (Full 38 vs Reduced 20)
# ---------------------------------------------------------
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

y_test_bin = label_binarize(y_test, classes=[1, 2, 3, 4, 5])
classes = [1, 2, 3, 4, 5]
n_classes = len(classes)

fig, axes = plt.subplots(1, 2, figsize=(18, 7.5))
esi_colors = ['#d62728', '#ff7f0e', '#2ca02c', '#1f77b4', '#9467bd']

def plot_roc_curves_on_ax(ax, probs, title_text):
    fpr, tpr, roc_aucs = dict(), dict(), dict()
    for i, cls in enumerate(classes):
        fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], probs[:, i])
        roc_aucs[i] = auc(fpr[i], tpr[i])
    
    fpr["micro"], tpr["micro"], _ = roc_curve(y_test_bin.ravel(), probs.ravel())
    roc_aucs["micro"] = auc(fpr["micro"], tpr["micro"])
    
    all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_classes)]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(n_classes):
        mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
    mean_tpr /= n_classes
    fpr["macro"] = all_fpr
    tpr["macro"] = mean_tpr
    roc_aucs["macro"] = auc(fpr["macro"], tpr["macro"])
    
    ax.plot(fpr["micro"], tpr["micro"], label=f"Micro-Average (AUC = {roc_aucs['micro']:.4f})", color='#e377c2', linestyle=':', linewidth=2.5)
    ax.plot(fpr["macro"], tpr["macro"], label=f"Macro-Average (AUC = {roc_aucs['macro']:.4f})", color='#17becf', linestyle='--', linewidth=2.5)
    
    for i in range(5):
        ax.plot(fpr[i], tpr[i], color=esi_colors[i], linewidth=2.0, label=f"ESI {i+1} (AUC = {roc_aucs[i]:.4f})")
    
    ax.plot([0, 1], [0, 1], 'k--', color='gray', linewidth=1.2, label='Random Guess (AUC = 0.5000)')
    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    ax.set_xlabel('False Positive Rate (1 - Specificity)', fontsize=11, fontweight='bold')
    ax.set_ylabel('True Positive Rate (Recall / Sensitivity)', fontsize=11, fontweight='bold')
    ax.set_title(title_text, fontsize=12, fontweight='bold', pad=10)
    ax.legend(loc="lower right", fontsize=9.5, frameon=True, framealpha=0.95)
    ax.grid(True, linestyle='--', alpha=0.4)

plot_roc_curves_on_ax(axes[0], probs_38, f"Pipeline A: Full 38 Features ROC Curves (Macro AUC = {report_38.loc[5, 'ROC_AUC']:.4f})")
plot_roc_curves_on_ax(axes[1], probs_20, f"Pipeline B: Reduced 20 Features ROC Curves (Macro AUC = {report_20.loc[5, 'ROC_AUC']:.4f})")

plt.tight_layout()
roc_comp_path = os.path.join(plots_dir, "oof_stacking_feature_ablation_roc_auc.png")
plt.savefig(roc_comp_path, dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(plots_dir, 'image', 'oof_stacking_feature_ablation_roc_auc.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Side-by-side ROC Curves saved to: {roc_comp_path}")

In [ ]:
# ---------------------------------------------------------
# Step 8: Export Feature Ablation Production Artifact Bundle & Metadata Manifest
# ---------------------------------------------------------
deploy_dir = f'{ROOT}/deploy'
os.makedirs(deploy_dir, exist_ok=True)

bundle = {
    'pipeline_38': {
        'scaler': scaler_38,
        'l1': l1_38,
        'l2': l2_38,
        'l3a': l3a_38,
        'l3b': l3b_38,
        'meta': meta_38,
        'n_features': 38
    },
    'pipeline_20': {
        'scaler': scaler_20,
        'l1': l1_20,
        'l2': l2_20,
        'l3a': l3a_20,
        'l3b': l3b_20,
        'meta': meta_20,
        'feature_names': feature_names_20,
        'n_features': 20
    }
}

bundle_file = os.path.join(deploy_dir, 'oof_stacking_feature_ablation_bundle.pkl')
with open(bundle_file, 'wb') as f:
    pickle.dump(bundle, f)

manifest = dict(
    pipeline='OOF_Stacking_Feature_Ablation_Benchmark',
    dataset='datasets/5v_cleandf.RData',
    n_classes=5,
    classes=['ESI 1', 'ESI 2', 'ESI 3', 'ESI 4', 'ESI 5'],
    holdout_test_samples=len(y_test),
    layer1_binary_metrics=l1_report_df.to_dict(orient='records'),
    layer2_binary_metrics=l2_report_df.to_dict(orient='records'),
    layer3a_binary_metrics=l3a_report_df.to_dict(orient='records'),
    layer3b_binary_metrics=l3b_report_df.to_dict(orient='records'),
    raw_chain_38_metrics=report_raw_38.to_dict(orient='records'),
    raw_chain_20_metrics=report_raw_20.to_dict(orient='records'),
    full_38_metrics=report_38.to_dict(orient='records'),
    reduced_20_metrics=report_20.to_dict(orient='records'),
    per_class_comparison=comp_rows
)

manifest_file = os.path.join(deploy_dir, 'oof_stacking_feature_ablation_manifest.json')
with open(manifest_file, 'w') as f:
    json.dump(manifest, f, indent=2)

print(f"✓ Comparison Bundle   : {bundle_file}")
print(f"✓ Comparison Manifest : {manifest_file}")